# 1-DOF MCP Static Model With A Nonlinear Passive Joint Law

### Question

How does the equilibrium behavior change if passive MCP resistance rises faster than linearly at larger flexion angles?


Notebook `01` established the clean linear baseline:

$$
\tau_{\mathrm{app}} = k(\theta - \theta_0)
$$

This notebook keeps the same overall setup:

- single MCP joint only
- planar flexion-extension only
- quasi-static torque balance only
- actuator represented as an applied joint torque

but changes one modeling assumption:

- passive joint torque is no longer purely linear


## Passive Torque Model

We use a simple cubic passive law:

$$
\tau_{\mathrm{passive}}(\theta) = -\left[k_1(\theta - \theta_0) + k_3(\theta - \theta_0)^3\right]
$$

So static equilibrium becomes:

$$
\tau_{\mathrm{app}} = k_1(\theta - \theta_0) + k_3(\theta - \theta_0)^3
$$

Interpretation:

- `k_1` is the small-angle linear stiffness term
- `k_3` makes resistance grow faster at larger angles
- if `k_3 = 0`, the model collapses back to the linear notebook


## Modeling Choice

This is a good first nonlinear step because it stays interpretable.

- it is still only 1 DOF
- it still uses a static torque balance
- the inverse relation is still easy to evaluate
- the forward relation needs numerical solving, which is the first useful new computational step


In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import root_scalar


def passive_torque_cubic(theta, k1, k3, theta_0=0.0):
    delta = theta - theta_0
    return -(k1 * delta + k3 * delta**3)


def required_torque_cubic(theta_target, k1, k3, theta_0=0.0):
    delta = theta_target - theta_0
    return k1 * delta + k3 * delta**3


def equilibrium_angle_cubic(tau_app, k1, k3, theta_0=0.0, bracket=(0.0, np.deg2rad(120.0))):
    def residual(theta):
        return required_torque_cubic(theta, k1=k1, k3=k3, theta_0=theta_0) - tau_app

    solution = root_scalar(residual, bracket=bracket, method="brentq")
    return solution.root


theta_0 = 0.0
k1 = 0.08  # N·m/rad
k3_values = np.array([0.00, 0.01, 0.02])  # N·m/rad^3
mcp_interpretation_window_deg = (0.0, 80.0)
angle_plot_range_deg = (0.0, 120.0)

tau_app = np.linspace(0.0, 0.12, 120)
theta_target_deg = np.linspace(angle_plot_range_deg[0], angle_plot_range_deg[1], 120)
theta_target_rad = np.deg2rad(theta_target_deg)

tau_required_by_k3 = {}
theta_eq_by_k3_deg = {}

for k3 in k3_values:
    tau_required_by_k3[k3] = required_torque_cubic(theta_target_rad, k1=k1, k3=k3, theta_0=theta_0)
    theta_eq = [equilibrium_angle_cubic(tau, k1=k1, k3=k3, theta_0=theta_0) for tau in tau_app]
    theta_eq_by_k3_deg[k3] = np.rad2deg(theta_eq)


## Inverse View

This plot is usually the easiest one to interpret first.

As `k_3` increases, the torque needed to hold larger angles increases faster than linearly.


In [2]:
fig = go.Figure()

for k3 in k3_values:
    label = f"k3 = {k3:.2f} N·m/rad^3"
    fig.add_trace(
        go.Scatter(
            x=theta_target_deg,
            y=tau_required_by_k3[k3],
            mode="lines",
            name=label,
            hovertemplate="theta_target = %{x:.1f} deg<br>tau_required = %{y:.3f} N·m<br>%{fullData.name}<extra></extra>",
        )
    )

fig.add_vrect(
    x0=mcp_interpretation_window_deg[0],
    x1=mcp_interpretation_window_deg[1],
    fillcolor="lightgreen",
    opacity=0.12,
    line_width=0,
    annotation_text="provisional interpretation band",
    annotation_position="top left",
)

fig.update_layout(
    title="1-DOF MCP Static Model With Cubic Passive Torque: Required Torque For A Target Angle",
    xaxis_title="Target MCP angle, theta_target [deg]",
    yaxis_title="Required actuator torque, tau_required [N·m]",
    legend_title_text="Nonlinear stiffness",
    template="plotly_white",
)

fig.update_xaxes(showgrid=True, range=list(angle_plot_range_deg))
fig.update_yaxes(showgrid=True)
fig.show()


## Forward View

Now the forward relation is no longer just a straight algebraic rearrangement.

For each applied torque, we solve the nonlinear equilibrium equation numerically.

This is the first real sign that even a very simple biomechanical model can remain conceptually clean while becoming computationally richer.


In [3]:
fig = go.Figure()

for k3 in k3_values:
    label = f"k3 = {k3:.2f} N·m/rad^3"
    fig.add_trace(
        go.Scatter(
            x=tau_app,
            y=theta_eq_by_k3_deg[k3],
            mode="lines",
            name=label,
            hovertemplate="tau_app = %{x:.3f} N·m<br>theta_eq = %{y:.1f} deg<br>%{fullData.name}<extra></extra>",
        )
    )

fig.add_hrect(
    y0=mcp_interpretation_window_deg[0],
    y1=mcp_interpretation_window_deg[1],
    fillcolor="lightgreen",
    opacity=0.12,
    line_width=0,
    annotation_text="provisional interpretation band",
    annotation_position="top left",
)

fig.update_layout(
    title="1-DOF MCP Static Model With Cubic Passive Torque: Equilibrium Angle Under Applied Torque",
    xaxis_title="Applied actuator torque, tau_app [N·m]",
    yaxis_title="Equilibrium MCP angle, theta_eq [deg]",
    legend_title_text="Nonlinear stiffness",
    template="plotly_white",
)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True, range=list(angle_plot_range_deg))
fig.show()


## What To Look For

- `k3 = 0` reproduces the linear baseline for this same `k1`
- larger `k3` bends the inverse curve upward
- larger `k3` flattens the forward torque-to-angle curve at larger flexion angles
- the model is still simple, but it now captures the idea that passive tissues may stiffen more strongly away from neutral
